# omega-cov calibration on Colab

Calibrates `THRESHOLD_ANTI` against TriviaQA + WikiBio using Mistral-7B-v0.1.

**Runtime > T4 GPU.** On CPU this is too slow to be useful.

**Required:** an Anthropic API key for the LLM judge step on TriviaQA.

Estimated wall time on T4: ~45 min for TriviaQA (n=1000) + ~30 min for WikiBio (n=500).

## 1. Install

In [ ]:
!git clone https://github.com/NovanBaillif/omega-cov.git
%cd omega-cov
!pip install -e ".[calibrate]" --quiet

## 2. Anthropic API key

Used by the TriviaQA judge step (Claude Haiku 4.5, T=0). Approx 1000 calls per run, ~\$0.30 total.

In [ ]:
import os, getpass
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

## 3. Hugging Face login (optional)

Mistral-7B-v0.1 is a gated model — you need a HF account that has accepted the license, and you need to be logged in. Skip this cell if you have already accepted via `huggingface-cli login` in a previous Colab session, or if you are using a non-gated alternative model.

In [ ]:
from huggingface_hub import login
login()

## 4. Calibrate on TriviaQA

In [ ]:
!python scripts/calibrate_trivia.py --n 1000 --seed 42

## 5. Calibrate on WikiBio

In [ ]:
!python scripts/calibrate_wikibio.py --n 500 --seed 42

## 6. Compute thresholds

Reads both CSVs, computes Youden's J per dataset, prints the recommendation, and saves `data/calibration_results.json`.

In [ ]:
!python scripts/analyze_thresholds.py

## 7. Download results

Pull the CSVs and the JSON summary back to your laptop, then commit the threshold update locally.

In [ ]:
from google.colab import files
files.download('data/trivia_results.csv')
files.download('data/wikibio_results.csv')
files.download('data/calibration_results.json')

## 8. Update `omega_cov/thresholds.py`

Locally, edit the constants based on the recommendation printed in step 6:

```python
THRESHOLD_ANTI = -0.027    # from calibration
THRESHOLD_ALIGNED = 0.1    # still placeholder, calibrate separately
```

Append a row to `docs/calibration.md` (Run log table) with the date, sample sizes, seed, optima, and the new commit hash. Then commit and push.